In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# DN = 'C://work/dev/python/progs/texts/sec_bert/'
DN = '/home/jovyan/work/sec_bert/'

import os
os.chdir(DN)

# Train

In [5]:
import pandas as pd
import numpy as np
import joblib
import json
from itertools import chain
import click
import os

from ruamel.yaml import YAML

# Загрузка данных

# train

In [ ]:
from src.train import train

train()

## pred

In [ ]:
from src.predict import predict

pred_df = predict(['APT32 compromised McAfee ePO to move laterally by distributing malware as a software deployment task.', 'Monitor executed commands and arguments that may indicate common cryptomining or proxyware functionality.', 'TeamTNT has created system services to execute cryptocurrency mining software',
        'An adversary may abuse configurations where an application has the setuid or setgid bits set in order to get code running in a different (and possibly more privileged) user’s context'])
pred_df

In [ ]:
s = 'Adversaries may leverage the resources of co-opted systems to complete resource-intensive tasks, which may impact system and/or hosted service availability.'
pred_main[pred_main.sentence.str.contains(s)]

In [6]:
from src.train import load_external_data, enc_classes

conf = YAML().load(open('params.yaml'))
df = load_external_data(conf)
df, mlb, mlb_ttp = enc_classes(df, conf, use_rare_ttp=False)

# на самом деле 208 train тут уже есть - синтетика
df['split'] = df['split'].fillna('tr')

In [8]:
from src.predict import predict
pred_main = predict(df['sentence'].to_numpy().tolist())
pred_main.head()

/home/jovyan/work/sec_bert/src/predict.py:38: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_bert_tak = torch.load(conf['train_fin']['model_taktic_fn'])
/home/jovyan/wo

,sentence,target,proba_tak,pred_tak,proba_tech,pred_tech,pred_str_tech,pred_str_tak
0,Adversaries may inject malicious code into pro...,1,"[0.9159165024757385, 0.945948600769043, 0.0647...","[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0.00013136507186573, 0.0001630313490750268, 5...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","(T1036.009, T1055, T1055.002, T1055.003, T1055...","(defense-evasion, privilege-escalation)"
1,"Before creating a window, graphical Windows-ba...",1,"[0.17777800559997559, 0.2613764703273773, 0.44...","[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0.00035170355113223195, 0.0001623184507479891...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",(),"(execution,)"
2,"Although small, the EWM is large enough to sto...",1,"[0.5367465615272522, 0.8571009635925293, 0.035...","[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0.0001453641161788255, 0.0001182870109914802,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","(T1055,)","(defense-evasion, privilege-escalation)"
3,Execution granted through EWM injection may al...,1,"[0.7098747491836548, 0.840006411075592, 0.2422...","[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0.00015069350774865597, 0.0001444405934307724...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","(T1036.009, T1055.002, T1055.003, T1055.004, T...","(defense-evasion, privilege-escalation)"
4,Running code in the context of another process...,1,"[0.9576749205589294, 0.6173551082611084, 0.033...","[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0.00013137972564436495, 8.765048551140353e-05...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","(T1036.009, T1055.002, T1055.003, T1055.005, T...","(defense-evasion, privilege-escalation)"


In [11]:
pred_main = pred_main.assign(target=df.target_ttp)


In [12]:

from sklearn.metrics import (average_precision_score, log_loss, confusion_matrix,
                            precision_recall_fscore_support, f1_score)

p_val_micro, r_val_micro, f1_val_micro, sup = precision_recall_fscore_support(np.array(pred_main['target'].values.tolist()), 
                                                    np.array(pred_main['pred_tech'].values.tolist()), average='micro')
p_val_macro, r_val_macro, f1_val_macro, sup = precision_recall_fscore_support(np.array(pred_main['target'].values.tolist()), 
                                                    np.array(pred_main['pred_tech'].values.tolist()), average='macro')

f1_val_micro, f1_val_macro

(0.37460134355703334, 0.2946893722338912)

## nttp pred

In [ ]:
df = load_external_data(conf)
df, mlb, mlb_ttp = enc_classes(df, conf, use_rare_ttp=False)

# на самом деле 208 train тут уже есть - синтетика
df['split'] = df['split'].fillna('tr')

In [ ]:
import torch

model_bert_ttp = torch.load(conf['train_fin']['model_technik_fn'])

model_bert = torch.load(conf['train_fin']['model_taktic_fn'])
    
thresh_ttp_l = joblib.load(conf['train_fin']['thresh_ttp_fn'])
thresh_l = joblib.load(conf['train_fin']['thresh_fn'])

In [ ]:
from src.predict import predict_bert

pred_df = predict_bert(df[['sentence']].assign(target=1), model_bert, thresh_l, conf_bert, suf='labels')
pred_df.head()

In [ ]:

pred_true_df = pred_df.copy().assign(target=df.target)

In [ ]:
from sklearn.metrics import (average_precision_score, log_loss, confusion_matrix,
                            precision_recall_fscore_support, f1_score)

p_val_micro, r_val_micro, f1_val_micro, sup = precision_recall_fscore_support(np.array(pred_true_df['target'].values.tolist()), 
                                                    np.array(pred_true_df['pred_labels'].values.tolist()), average='micro')
p_val_macro, r_val_macro, f1_val_macro, sup = precision_recall_fscore_support(np.array(pred_true_df['target'].values.tolist()), 
                                                    np.array(pred_true_df['pred_labels'].values.tolist()), average='macro')

f1_val_micro, f1_val_macro

## ttp pred

In [ ]:
pred_df = predict_bert(pred_df, model_bert_ttp, thresh_ttp_l, conf_bert_ttp, suf='ttp')
pred_df.head()

In [ ]:
pred_true_df = pred_df.copy().assign(target=df.target_ttp)


In [ ]:
from sklearn.metrics import (average_precision_score, log_loss, confusion_matrix,
                            precision_recall_fscore_support, f1_score)

p_val_micro, r_val_micro, f1_val_micro, sup = precision_recall_fscore_support(np.array(pred_true_df['target'].values.tolist()), 
                                                    np.array(pred_true_df['pred_ttp'].values.tolist()), average='micro')
p_val_macro, r_val_macro, f1_val_macro, sup = precision_recall_fscore_support(np.array(pred_true_df['target'].values.tolist()), 
                                                    np.array(pred_true_df['pred_ttp'].values.tolist()), average='macro')

f1_val_micro, f1_val_macro

In [ ]:
from sklearn.metrics import (average_precision_score, log_loss, confusion_matrix,
                            precision_recall_fscore_support, f1_score)

p_val_micro, r_val_micro, f1_val_micro, sup = precision_recall_fscore_support(np.array(pred_true_df['target'].values.tolist()), 
                                                    np.array(pred_true_df['pred_ttp'].values.tolist()), average='micro')
p_val_macro, r_val_macro, f1_val_macro, sup = precision_recall_fscore_support(np.array(pred_true_df['target'].values.tolist()), 
                                                    np.array(pred_true_df['pred_ttp'].values.tolist()), average='macro')

f1_val_micro, f1_val_macro

# Добавим полей с метками


In [ ]:
pred_df['pred_str_ttp'] = pred_df['pred_ttp'].map(lambda x: mlb_ttp.inverse_transform(np.array([x]))[0])
pred_df['pred_str_labels'] = pred_df['pred_labels'].map(lambda x: mlb.inverse_transform(np.array([x]))[0])

pred_df = pred_df.assign(true_labels=df.labels, true_ttp=df.ttp)

In [ ]:
pred_df